# CI/CD & Automation with Databricks

Declarative Automation Bundles (DABs), Git Folders, and automated deployment patterns.

| Topic | Key Concept | Exam Keywords |
|-------|-------------|---------------|
| DABs | YAML-based job/pipeline definitions | `databricks.yml`, `bundle deploy` |
| Git Folders | Repo-backed workspaces | version control, CI/CD integration |
| Target Overrides | Per-environment config | dev, staging, prod |
| CLI Deploy | `databricks bundle deploy` | automated deployment |

## Learning Objectives

After completing this module you will be able to:

- Explain the role of **Declarative Automation Bundles (DABs)** in CI/CD workflows
- Read and write a `databricks.yml` bundle configuration
- Use **Git Folders** to connect a workspace to a Git repository
- Configure **target overrides** for dev / staging / prod environments
- Deploy a bundle using the **Databricks CLI**

## CI/CD Overview for Data Engineering

**Why CI/CD matters:**
- Reproducible deployments across environments
- Code review via pull requests
- Automated testing before promotion
- Rollback capability

**Databricks CI/CD tools:**

| Tool | Purpose | Storage |
|------|---------|--------|
| Git Folders | Notebook version control | Git repo (GitHub, GitLab, ADO) |
| DABs (Declarative Automation Bundles) | Job/pipeline YAML definitions | `databricks.yml` |
| Databricks CLI | Deploy, run, validate bundles | Local machine / CI runner |

> **Exam Tip:** DABs replaced the older Databricks Asset Bundles terminology — same YAML format.

## Git Folders

**Git Folders** (formerly Repos) connect a Databricks workspace folder to a Git repository.

### Setup Workflow:
1. **Workspace** → Git Folders → Add a Folder
2. Enter the repository URL (GitHub / GitLab / ADO)
3. Choose the branch (e.g., `main`, `develop`)
4. **Pull** to sync latest changes

### Key Operations:

| Action | Description |
|--------|-------------|
| `git pull` | Sync remote changes to workspace |
| Create branch | Work in isolation, then merge via PR |
| Commit & push | Save notebook changes back to Git |
| Git diff | Compare notebook versions |

### Exam Tip:
> Git Folders enable **source control** for notebooks in Databricks.
> Changes can be committed directly from the workspace UI or via the Databricks CLI.

## Declarative Automation Bundles (DABs)

**DABs** define Databricks resources (jobs, pipelines, clusters) as YAML configuration files.

### Bundle Structure:
```
my_bundle/
├── databricks.yml          # Root config: bundle name, workspace, targets
├── resources/
│   ├── job_medallion.yml   # Job definition
│   └── pipeline_orders.yml # Lakeflow pipeline definition
└── src/
    └── notebooks/          # Source notebooks referenced by jobs
```

### Core DABs Commands:

| Command | Action |
|---------|--------|
| `databricks bundle validate` | Validate YAML syntax |
| `databricks bundle deploy` | Deploy resources to workspace |
| `databricks bundle run <job>` | Run a deployed job |
| `databricks bundle destroy` | Remove all deployed resources |

> **Exam Tip:** `databricks bundle deploy --target prod` deploys the `prod` target configuration.

### Declarative Automation Bundles (DABs) — YAML Configuration

Below is the **Declarative Automation Bundles** (formerly Databricks Asset Bundles / DABs) YAML definition for the Medallion pipeline Job. This was exported from a working Databricks deployment.

> **Before deploying:** Update `notebook_path`, `existing_cluster_id`, and `parameters` to match your environment.

```yaml
resources:
  jobs:
    job_demo_2:
      name: job_demo_medalion
      tasks:
        - task_key: bronze_customer
          notebook_task:
            notebook_path: /Workspace/Users/<your_user>/repo/databricks-lakehouse-transformation/materials/medallion/bronze_customers
            source: WORKSPACE
          environment_key: Default
        - task_key: bronze_orders
          notebook_task:
            notebook_path: /Workspace/Users/<your_user>/repo/databricks-lakehouse-transformation/materials/medallion/bronze_orders
            source: WORKSPACE
          environment_key: Default
        - task_key: silver_customer
          depends_on:
            - task_key: bronze_customer
          notebook_task:
            notebook_path: /Workspace/Users/<your_user>/repo/databricks-lakehouse-transformation/materials/medallion/silver_customers
            source: WORKSPACE
          environment_key: Default
        - task_key: silver_orders_cleaned
          depends_on:
            - task_key: bronze_orders
          notebook_task:
            notebook_path: /Workspace/Users/<your_user>/repo/databricks-lakehouse-transformation/materials/medallion/silver_orders_cleaned
            source: WORKSPACE
          environment_key: Default
        - task_key: gold_daily_orders
          depends_on:
            - task_key: silver_orders_cleaned
          notebook_task:
            notebook_path: /Workspace/Users/<your_user>/repo/databricks-lakehouse-transformation/materials/medallion/gold_daily_orders
            source: WORKSPACE
          environment_key: Default
        - task_key: gold_customer_orders_summary
          depends_on:
            - task_key: silver_customer
            - task_key: gold_daily_orders
          notebook_task:
            notebook_path: /Workspace/Users/<your_user>/repo/databricks-lakehouse-transformation/materials/medallion/gold_customer_orders_summary
            source: WORKSPACE
          environment_key: Default
      queue:
        enabled: true
      parameters:
        - name: catalog
          default: retailhub_trainer
        - name: source_path
          default: /Volumes/retailhub_trainer/default/datasets
      environments:
        - environment_key: Default
          spec:
            environment_version: "5"
      performance_target: PERFORMANCE_OPTIMIZED

```

| Element | Description |
|---------|-------------|
| `existing_cluster_id` | Use existing all-purpose cluster (for demo) or replace with `job_cluster_key` for production |
| `depends_on` | Defines task dependencies — creates the DAG |
| `parameters` | Job-level parameters passed to all notebooks via `dbutils.widgets` |
| `queue.enabled` | Queues runs if max concurrent reached |
| `source: WORKSPACE` | Notebook is in Workspace (not Repos) |

## Target Overrides — Multi-Environment Configuration

DABs support **targets** to configure environment-specific settings (dev / staging / prod):

```yaml
bundle:
  name: medallion_pipeline

workspace:
  host: https://adb-<workspace-id>.azuredatabricks.net

targets:
  dev:
    mode: development          # enables 'dev_' prefix on resource names
    workspace:
      host: https://adb-dev.azuredatabricks.net
    variables:
      catalog: dev_catalog
      schema: bronze

  prod:
    mode: production
    workspace:
      host: https://adb-prod.azuredatabricks.net
    variables:
      catalog: prod_catalog
      schema: bronze
```

| Target mode | Behavior |
|-------------|----------|
| `development` | Adds `[dev <username>]` prefix; paused triggers |
| `production` | No prefix; live triggers; no run-as override |

> **Exam Tip:** `mode: development` automatically pauses all job triggers — safe for testing.

### Deploy Checklist

**Option A — JSON Editor (UI):**
1. [ ] Workflows → Create Job
2. [ ] Click ` Edit as JSON` (top-right)
3. [ ] Paste the JSON above (replace placeholders)
4. [ ] Save → Run now

**Option B — Databricks CLI:**
```bash
# Save JSON to file, then:
databricks jobs create --json @medallion_pipeline_job.json
```

**After deployment — show participants:**
- [ ] DAG visualization (fan-out at Bronze, fan-in at Gold)
- [ ] Task-level parameters (catalog, schema, source_path)
- [ ] Trigger: set to PAUSED — run manually for demo
- [ ] Run → observe sequential layer execution (Bronze → Silver → Gold)
- [ ] Show Repair Run: intentionally fail one task → repair reruns only failed + downstream

## Databricks CLI — Setup and Authentication

```bash
# Install Databricks CLI v0.200+
pip install databricks-cli

# Or install new Go-based CLI
brew install databricks/tap/databricks

# Configure authentication
databricks configure --token
# Enter host: https://adb-<workspace-id>.azuredatabricks.net
# Enter token: <personal-access-token>

# --- Bundle workflow ---
cd my_bundle/

# Validate YAML
databricks bundle validate

# Deploy to dev target
databricks bundle deploy --target dev

# Deploy to production
databricks bundle deploy --target prod

# Run a deployed job
databricks bundle run medallion_job --target prod
```

## CI Pipeline Integration (GitHub Actions / Azure DevOps)

```yaml
# .github/workflows/deploy.yml
name: Deploy Medallion Pipeline

on:
  push:
    branches: [main]

jobs:
  deploy:
    runs-on: ubuntu-latest
    steps:
      - uses: actions/checkout@v3
      - name: Install Databricks CLI
        run: pip install databricks-cli
      - name: Deploy bundle
        env:
          DATABRICKS_HOST: ${{ secrets.DATABRICKS_HOST }}
          DATABRICKS_TOKEN: ${{ secrets.DATABRICKS_TOKEN }}
        run: databricks bundle deploy --target prod
```

> **Pattern:** Push to `main` → CI validates → `bundle deploy --target prod` → resources updated automatically.

## Summary

| Topic | Key Concept | Exam Keywords |
|-------|-------------|---------------|
| Git Folders | Notebook version control in workspace | `git pull`, branch, PR |
| DABs | YAML-defined jobs and pipelines | `databricks.yml`, bundle |
| Target Overrides | Per-environment config | `dev`, `prod`, `mode: development` |
| CLI Deploy | `bundle deploy --target` | `databricks bundle deploy` |
| CI Integration | GitHub Actions / Azure DevOps | automated on push to main |

← [08 — Job Orchestration](08_job_orchestration.ipynb) | **[ README](../../../README.md)** | [10 — Governance & Security →](10_governance_and_security.ipynb)